# bn-weight-bias-init-pattern — faded example 3: Fill the isinstance guard for BN layers

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bn-weight-bias-init-pattern`. The last cell reports your progress on the `GAN: BN weight=1 bias=0 init` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BN weight=1 bias=0 init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bn-weight-bias-init-pattern`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bn-weight-bias-init-pattern"
DD_SUBTOPIC = "GAN: BN weight=1 bias=0 init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The DCGAN BN init must run only on BatchNorm submodules. The `isinstance` guard inside `init_fn` is what protects Conv and Linear weights from being overwritten by the BN scheme during `model.apply`.

## Faded exercise 3

### Faded — fill the isinstance guard

The init body (resample gamma, zero beta) is written for you. Complete the ONE condition that decides *which* modules receive the init — it must match BatchNorm1d and BatchNorm2d and nothing else.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn as nn

t.manual_seed(0)

def init_fn(m):
    is_bn = isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))
    if is_bn:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

model = nn.Sequential(nn.Conv2d(3, 6, 3), nn.BatchNorm2d(6), nn.Linear(6, 2))
conv_before = model[0].weight.clone()
model.apply(init_fn)


def _test():
    import torch.nn as nn
    # Conv weight must be untouched by the BN init
    assert t.equal(conv_before, model[0].weight), "non-BN (Conv) weight must be unchanged"
    # BN must have been initialized: beta zeroed, gamma resampled off the constant 1
    bn = model[1]
    assert t.allclose(bn.bias, t.zeros_like(bn.bias)), "BN bias must be zeroed"
    assert not t.allclose(bn.weight, t.ones_like(bn.weight)), "BN gamma must be resampled"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

t.manual_seed(0)

def init_fn(m):
    is_bn = isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))
    if is_bn:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

model = nn.Sequential(nn.Conv2d(3, 6, 3), nn.BatchNorm2d(6), nn.Linear(6, 2))
conv_before = model[0].weight.clone()
model.apply(init_fn)
```
</details>